# Causal Effect & Concept Drift Analysis
**Mục tiêu:** Áp dụng hướng tiếp cận của Jan Niklas Adams để:
1. Phát hiện sự thay đổi cấu trúc của quy trình theo thời gian (Concept Drift).
2. Xác định nguyên nhân gốc rễ (Root Cause) gây ra sự thay đổi đó bằng Suy luận nhân quả (Causal Inference).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Phát hiện Concept Drift theo thời gian
Chúng ta sẽ phân tích xem tỷ lệ hồ sơ bị thiếu giấy tờ (`A_Incomplete`) có bị thay đổi đột ngột tại một thời điểm nào đó trong năm 2016-2017 hay không.

In [ ]:
# Đọc dữ liệu event log
print("Loading data...")
df = pd.read_csv('../data/bpi-challenge-2017/bpi_2017_cleaned.csv')
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True, errors='coerce')

# Trích xuất thời điểm bắt đầu của từng hồ sơ và số lượng event A_Incomplete
case_starts = df.groupby('case:concept:name')['time:timestamp'].min().reset_index(name='start_time')
incomplete_events = df[df['concept:name'] == 'A_Incomplete'].groupby('case:concept:name').size().reset_index(name='incomplete_count')

# Merge dữ liệu
cases = case_starts.merge(incomplete_events, on='case:concept:name', how='left').fillna({'incomplete_count': 0})
cases['has_incomplete'] = (cases['incomplete_count'] > 0).astype(int)

# Nhóm theo tuần để xem xu hướng (Time-series Aggregation)
cases['start_week'] = cases['start_time'].dt.to_period('W').dt.start_time
weekly_stats = cases.groupby('start_week').agg(
    total_cases=('case:concept:name', 'count'),
    incomplete_rate=('has_incomplete', 'mean'),
    avg_incomplete_count=('incomplete_count', 'mean')
).reset_index()

display(weekly_stats.head())

### Trực quan hóa để tìm Change Point (Điểm gãy)

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(weekly_stats['start_week'], weekly_stats['incomplete_rate'], marker='o', linestyle='-', color='b', label='Tỷ lệ hồ sơ bị A_Incomplete')

# Vẽ đường trung bình trượt (Moving Average) để nhìn rõ xu hướng
weekly_stats['ma_4'] = weekly_stats['incomplete_rate'].rolling(window=4).mean()
plt.plot(weekly_stats['start_week'], weekly_stats['ma_4'], color='r', linewidth=2, label='Trung bình trượt (4 tuần)')

plt.title('Phát hiện Concept Drift: Tỷ lệ thiếu hồ sơ theo thời gian', fontsize=14)
plt.xlabel('Thời gian (Tuần)', fontsize=12)
plt.ylabel('Tỷ lệ hồ sơ bị A_Incomplete', fontsize=12)
plt.legend()
plt.show()